# 07 – Person Alternative Names

Esplorazione e data cleaning del dataset `person_alternate_names.csv`.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `person_mal_id` | ID univoco della persona su MyAnimeList |
| `alt_name` | Nome alternativo della persona |

## 1. Import e caricamento dati
Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [ ]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze
from foreign_key_analyzer import check_fk

df_alt = pd.read_csv('../datasets/person_alternate_names.csv')
print(f'Shape: {df_alt.shape}')
print()
df_alt.info()
print()
df_alt.head()

**Osservazioni iniziali:**
- Il dataset contiene **20.465 righe** e **2 colonne**.
- `person_mal_id` è completo: **nessun valore nullo**.
- `alt_name` presenta **18 valori nulli** (20.447 non-null su 20.465), da gestire in fase di cleaning.
- I tipi di dati sono adeguati: `int64` e `str`.

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [ ]:
n_originale = len(df_alt)

mask_dup = df_alt.duplicated(keep=False)
n_righe_coinvolte = mask_dup.sum()
n_gruppi = df_alt[mask_dup].duplicated(keep='first').sum()
n_tenute = n_righe_coinvolte - n_gruppi

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_alt.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_alt):,}')

**Osservazioni:**

**68 righe** (0.33% del totale) risultano coinvolte in duplicazioni esatte su entrambe le colonne. Di queste, **29** sono prime occorrenze mantenute e **39** occorrenze extra rimosse. Dopo la rimozione il dataset scende da 20.465 a **20.426 righe**.

Adesso che siamo sicuri che tutte le righe sono uniche, iniziamo l'analisi per colonne utilizzando la nostra libreria `dataset_analyzer`.

## 2. Analisi colonna per colonna

### 2.1 `person_mal_id`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria di `person_details.csv`.

I valori duplicati sono **attesi**: la stessa persona può avere più nomi alternativi.

I controlli rilevanti sono:
- **Valori nulli**: una chiave esterna nulla indica una riga senza riferimento che va rimossa.
- **Integrità referenziale**: ogni ID presente qui deve esistere in `person_details_clean.csv`.

Usiamo quindi `check_fk` al posto di `analyze`, che effettua entrambi i controlli.

In [ ]:
df_persons = pd.read_csv('../datasets_cleaned/person_details_clean.csv')

mask_orphan = check_fk(df_alt['person_mal_id'], df_persons['person_mal_id'], child_df=df_alt
)

print(f'Null in person_mal_id               : {df_alt["person_mal_id"].isna().sum()}')
print(f'Duplicati in person_mal_id (attesi) : {df_alt["person_mal_id"].duplicated().sum():,}')

**Osservazioni:**
- **Nessun valore nullo**: tutti i record hanno un ID di riferimento valido.
- **Integrità referenziale**: non ci sono righe orfane.

**Nessuna pulizia è necessaria.**